# P1 — YOLOv8n Fine-Tuning on Colab Pro

This notebook trains YOLOv8n on the custom household object dataset.

**Before running:**
1. Runtime → Change runtime type → **A100 GPU** (Colab Pro)
2. Upload `data/dataset/` to your Google Drive, update `DATASET_PATH` below
3. Set your W&B API key in the cell below

**After training:**
- Best weights are at `runs/detect/<RUN_NAME>/weights/best.pt`
- Download and copy to `models/best.pt` in this repo
- Then run `export.py` to get ONNX + blob

In [ ]:
# ── 1. Check GPU ──────────────────────────────────────────────────────────────
!nvidia-smi

In [ ]:
# ── 2. Install dependencies ────────────────────────────────────────────────────
!pip install -q ultralytics wandb

In [ ]:
# ── 3. Mount Google Drive ──────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 4. Config — edit these ────────────────────────────────────────────────────
# Path to your dataset.yaml inside Google Drive
DATASET_YAML = "/content/drive/MyDrive/oakd-vision-ml/data/dataset/dataset.yaml"

RUN_NAME  = "yolov8n-custom-v1"
PROJECT   = "oakd-vision-ml"
EPOCHS    = 200
IMGSZ     = 640
BATCH     = 16   # drop to 8 if OOM

In [ ]:
# ── 5. W&B login ──────────────────────────────────────────────────────────────
import wandb
wandb.login()  # paste your API key from wandb.ai/authorize

In [ ]:
# ── 6. Train ──────────────────────────────────────────────────────────────────
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # downloads COCO-pretrained weights automatically

results = model.train(
    data=DATASET_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project=PROJECT,
    name=RUN_NAME,
    # Augmentation
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    flipud=0.0, fliplr=0.5,
    mosaic=1.0, mixup=0.1,
    degrees=5.0, translate=0.1, scale=0.5,
    # Logging
    save=True, save_period=20, plots=True,
    patience=50,
)

best_pt = f"{PROJECT}/{RUN_NAME}/weights/best.pt"
print(f"Best weights: {best_pt}")

In [ ]:
# ── 7. Evaluate on test split ─────────────────────────────────────────────────
metrics = model.val(data=DATASET_YAML, split="test", plots=True)

print(f"mAP@50    : {metrics.box.map50:.4f}  (target ≥ 0.70)")
print(f"mAP@50:95 : {metrics.box.map:.4f}")
print(f"Precision : {metrics.box.mp:.4f}")
print(f"Recall    : {metrics.box.mr:.4f}")

In [ ]:
# ── 8. Copy best.pt to Drive ──────────────────────────────────────────────────
import shutil, os

drive_models = "/content/drive/MyDrive/oakd-vision-ml/models"
os.makedirs(drive_models, exist_ok=True)
shutil.copy(best_pt, f"{drive_models}/best.pt")
print("Saved to Drive:", f"{drive_models}/best.pt")